# Drug Target Enrichment via ChEMBL

Demonstrates `chemblDrugEnrichment.drug_enrichemnt_from_evidence` on a randomly
generated `(targetId, diseaseId, resourceScore)` evidence table, using the real
ChEMBL and disease-ontology parquets from this repo.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as f

from gentropy.common.session import Session
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment


In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "8G"})


## Load real data


In [ ]:
REPO_ROOT = ".."  # adjust if running from a different location

chembl_evidence = session.spark.read.parquet(f"{REPO_ROOT}/data/25.06/output/evidence/sourceId=chembl")

disease_index = session.spark.read.parquet(f"{REPO_ROOT}/data/25.06/output/disease/disease.parquet")

print(f"ChEMBL rows: {chembl_evidence.count():,}")
print(f"Disease nodes: {disease_index.count():,}")


## Generate random genetic evidence

Simulates a `(targetId, diseaseId, resourceScore)` table — the format expected by
`drug_enrichemnt_from_evidence`. Half the targets are drawn from the real ChEMBL
target pool so the enrichment signal is detectable.


In [ ]:
rng = np.random.default_rng(42)

# Pull target and disease ID pools from the real data
chembl_targets = [r["targetId"] for r in chembl_evidence.select("targetId").distinct().collect()]
all_diseases = [r["id"] for r in disease_index.select("id").collect()]

# Fake background genes not in ChEMBL
background_targets = [f"ENSG{i:011d}" for i in range(1, 3001)]

n = 3000
targets = list(rng.choice(chembl_targets, size=n // 2, replace=True)) + list(
    rng.choice(background_targets, size=n // 2, replace=True)
)
diseases = list(rng.choice(all_diseases, size=n, replace=True))
scores = rng.uniform(0.0, 1.0, size=n).tolist()

evidence_pd = pd.DataFrame({"targetId": targets, "diseaseId": diseases, "resourceScore": scores}).drop_duplicates(
    ["targetId", "diseaseId"]
)
print(f"Random evidence pairs: {len(evidence_pd):,}")
evidence_pd.head()


In [ ]:
# Convert to Spark DataFrame
evidence_spark = session.spark.createDataFrame(evidence_pd)


## Run enrichment


In [ ]:
# Evidence (evid) is the table of targetId-diseaseId-resourceScore trios.
#  Put 1 as teh resourceScore if all of them are significant.


In [ ]:
results = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence_spark,
    disease_index_orig=disease_index,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"],  # exclude oncology
)
results


## Plot


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

x = np.arange(len(results))
odds = results["odds_ratio"].values
ci_lo = results["ci_low"].values
ci_hi = results["ci_high"].values
yerr = np.vstack([odds - ci_lo, ci_hi - odds])

ax.bar(x, odds, color="steelblue", alpha=0.7, width=0.5)
ax.errorbar(x, odds, yerr=yerr, fmt="none", ecolor="black", capsize=5, linewidth=1.5)
ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5, label="Null (OR=1)")

for i, row in results.iterrows():
    ax.text(i, row["ci_high"] + 0.04, f"p={row['p_value']:.1e}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(results["clinicalPhase"].tolist(), fontsize=11)
ax.set_ylabel("Odds Ratio", fontsize=11)
ax.set_xlabel("Clinical phase threshold", fontsize=11)
ax.set_title("Drug target enrichment (random genetic evidence)", fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()
